# 0. Contexto del dataset.

#####Este dataset está ya limpio y analizado que está basándose en uno generado con un modelo de deep learning (train.csv, 188533 filas) basándose en un dataset con información sobre automóviles (used_cars.csv, 4009 filas). Ambos se pueden obtener aquí: https://www.kaggle.com/competitions/playground-series-s4e9/data

#####Las columnas que contiene son las siguientes:
- Brand y Model: indica la marca y el modelo específico.
- Model_year: indica el año de fabricación del vehículo.
- Mileage: indica el kilometraje (en millas) de cada vehículo.
- Fuel_type: indica si el vehículo es de gasolina, diesel, eléctrico o híbrido.
- Engine_type: indica las especificaciones del motor.
- Transmission: indica el tipo de transmisión, si es automático, manual o de alguna variante.
- Ext_col e int_col: indica el color interior y exterior.
- Accident: indica si el vehículo tiene un historial de accidentes o daños.
- Clean_title: indica si el vehículo ha sido dañado o reconstruido.
- Price: indica el precio de cada vehículo.
- Price_group: divide la columna de precios en 4 categorías.
- Milage_group: divide la columna de kilometraje en 3 categorías.

# 1. Se importan las librerías esenciales.

In [ ]:
#Importamos librerías esenciales
import pandas as pd # Para manejo de datos
import numpy as np # Para operaciones numéricas


from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import randint

import joblib #Para exportar los datos a streamlit

import re #Para expresiones regulares (reg exp)
import warnings
warnings.filterwarnings("ignore")



# 2. Se carga el dataset.

In [ ]:
df_train = pd.read_csv('limpieza_eda_P7_grupo2.csv')

In [ ]:
#Nos aseguramos de que sea un dataframe
df_train = pd.DataFrame(df_train)

In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 188533 entries, 0 to 188532
Data columns (total 17 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            188533 non-null  int64  
 1   brand         188533 non-null  object 
 2   model         188533 non-null  object 
 3   model_year    188533 non-null  int64  
 4   milage        188533 non-null  int64  
 5   fuel_type     188533 non-null  object 
 6   engine        188533 non-null  object 
 7   transmission  188533 non-null  object 
 8   ext_col       188533 non-null  object 
 9   int_col       188533 non-null  object 
 10  accident      188533 non-null  object 
 11  clean_title   188533 non-null  object 
 12  price         188533 non-null  int64  
 13  engine_hp     188533 non-null  float64
 14  engine_l      188533 non-null  float64
 15  price_group   188483 non-null  object 
 16  milage_group  188525 non-null  object 
dtypes: float64(2), int64(4), object(11)
memory usage

#3. Se obtiene una muestra.

In [ ]:
# Tomar muestra aleatoria del 50%
df_muestra = df_train.sample(frac=0.5, random_state=42)

#4. Modelo y entrenamiento

Con la limpieza y el EDA queda claro que la relación entre las columnas no es lineal, por lo que se usa un modelo no lineal como lo es el Random Forest.

No vamos a usar KNN porque el dataset tiene muchas filas y eso haría que no diera resultados exactos y exige gran potencia computacional.

##4.1. Random Forest sin ajuste de parámetros.


In [ ]:
# ========= FEATURES ========= #
features = [
    "brand",
    "model_year",
    "milage",
    "accident",
    "fuel_type",
    "clean_title",
    "price_group",
    "milage_group",
    "engine_hp",
    "engine_l"
]

#Se define X e y
X = df_muestra[features] #Variables que se quieren usar para predecir
y = df_muestra["price"] #Variable que se quiere predecir

print(f"Total muestras: {len(X)}")
print(f"Características: {X.shape[1]}")

Total muestras: 94266
Características: 10


In [ ]:
# ========= SPLIT ========= #
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#80% para entrenamiento y 20% para test
print(f"Train: {len(X_train)} muestras ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test: {len(X_test)} muestras ({len(X_test)/len(X)*100:.0f}%)")

Train: 75412 muestras (80%)
Test: 18854 muestras (20%)


In [ ]:
# ========= PREPROCESADO ========= #

cat_cols = ["brand", "clean_title", "accident", "fuel_type", "price_group", "milage_group"] #Variables categóricas
num_cols = ["model_year", "milage", "engine_hp", "engine_l" ] #Variables numéricas

preprocesado = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
])


In [ ]:
# ========= MODELO ========= #
modelo = Pipeline([
    ("prep", preprocesado),
    ("rf", RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_split=3,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42
    ))
])

In [ ]:
# ========= ENTRENAR ========= #
modelo.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['model_year', 'milage',
                                                   'engine_hp', 'engine_l']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['brand', 'clean_title',
                                                   'accident', 'fuel_type',
                                                   'price_group',
                                                   'milage_group'])])),
                ('rf',
                 RandomForestRegressor(max_depth=20, max_features='sqrt',
                                       min_samples_split=3, random_state=42))])

In [ ]:
# ========= PREDICCIONES ========= #
pred = modelo.predict(X_test)

In [ ]:
# ========= MÉTRICAS ========= #
r2_sinajuste = r2_score(y_test, pred)
MAE_sinajuste = mean_absolute_error(y_test, pred)
RMSE_sinajuste = np.sqrt(mean_squared_error(y_test, pred))

print("R² del modelo:", r2_sinajuste)
print("MAE del modelo:", MAE_sinajuste)
print("RMSE del modelo:", RMSE_sinajuste)

#INTERPRETACIÓN
print("\n" + "="*60)
print("=== INTERPRETACIÓN ===")
print("="*60)

print("""


MSE (Error Cuadrático Medio):
• Mide el error promedio al cuadrado
• Menor valor = mejor modelo
• Sensible a outliers (errores grandes pesan más)

R² (Coeficiente de Determinación):
• Mide cuánta variabilidad explica el modelo
• Rango: 0 a 1 (idealmente cerca de 1)
• 0 = modelo no explica nada
• 1 = modelo explica toda la variabilidad
• Puede ser negativo si el modelo es peor que usar la media""")

R² del modelo: 0.5527623450712703
MAE del modelo: 13271.934855237418
RMSE del modelo: 59069.80504210864

=== INTERPRETACIÓN ===



MSE (Error Cuadrático Medio):
• Mide el error promedio al cuadrado
• Menor valor = mejor modelo
• Sensible a outliers (errores grandes pesan más)

R² (Coeficiente de Determinación):
• Mide cuánta variabilidad explica el modelo
• Rango: 0 a 1 (idealmente cerca de 1)
• 0 = modelo no explica nada
• 1 = modelo explica toda la variabilidad
• Puede ser negativo si el modelo es peor que usar la media


Ahora buscaremos una manera de mejorar este modelo y obtener métricas mejores.

##4.2. Random Forest optimizado

Como el modelo depende de la "suerte" al dividir el dataset, al usar GridSearchCV o RandomizedSearchCV se optimizarán hiperparámetros y con la validación cruzada, al hacer varias iteraciones y la media de cada fold, se consigue que el azar juegue un papel menos importante a la hora de usar el modelo.

###4.2.1. Usando RandomisedSearchCV

In [ ]:
# ========= FEATURES ========= #
'''features = [
    "brand",
    "model_year",
    "milage",
    "fuel_type",
    "clean_title",
    "price_group",
    "milage_group",
    "engine_hp",
    "engine_l"
]
'''
features = [col for col in df_muestra.columns if col not in ['price', 'accident', 'brand', 'fuel_type','id', 'model', 'engine', 'ext_col', 'int_col']]
#Quitamos esas columnas porque no las creemos importantes para este modelo.
#Se dejan arriba las columnas en comentario para saber de un vistazo cuál tenemos.


X = df_muestra[features] #variables con las que se quiere predecir
y = df_muestra["price"] #variable que se quiere predecir

cat_cols = ["clean_title", "price_group", "milage_group"]
num_cols = ["model_year", "milage", "engine_hp", "engine_l"]

print(f"Total muestras (filas): {len(X)}")
print(f"Características (columnas): {X.shape[1]}")

Total muestras (filas): 94266
Características (columnas): 8


In [ ]:
# ========= SPLIT ========= #
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#80% para entrenamiento y 20% para test
print(f"Train: {len(X_train)} muestras ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test: {len(X_test)} muestras ({len(X_test)/len(X)*100:.0f}%)")

Train: 75412 muestras (80%)
Test: 18854 muestras (20%)


In [ ]:
# ========= PREPROCESADO ========= #
preprocesado = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
])

In [ ]:
# ========= MODELO ========= #
#Modelo base sin hiperparámetros a optimizar
modelo = Pipeline([
    ("prep", preprocesado),
    ("rf", RandomForestRegressor(
        random_state=42
    ))
])

In [ ]:
#Rejilla con hiperparámetros a ajustar
param_dist = {
    "rf__n_estimators": [100, 150],
    "rf__max_depth": [10, 20],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", "log2"]
}

In [ ]:
#Creación del RandomizedSearchCV con validación cruzada

scoring = {
    'R2': 'r2',
    'RMSE': 'neg_root_mean_squared_error',
    'MAE': 'neg_mean_absolute_error'
}

random_search = RandomizedSearchCV(
    estimator=modelo,
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring='r2', #elige el mejor modelo con el R2 mayor
    n_jobs=-1,
    random_state=42
)


In [ ]:
# Entrenar el modelo
random_search.fit(X_train, y_train)

print("Mejores parámetros:", random_search.best_params_)
print("Mejor score R²:", random_search.best_score_)
pred = random_search.predict(X_test)

Mejores parámetros: {'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 1, 'rf__max_features': 'sqrt', 'rf__max_depth': 10}
Mejor score R²: 0.6198599638943856


In [ ]:
#Muestra los que más se ajustan
print(random_search.best_params_)
print(random_search.best_estimator_)

{'rf__n_estimators': 100, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 1, 'rf__max_features': 'sqrt', 'rf__max_depth': 10}
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['model_year', 'milage',
                                                   'engine_hp', 'engine_l']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['clean_title', 'price_group',
                                                   'milage_group'])])),
                ('rf',
                 RandomForestRegressor(max_depth=10, max_features='sqrt',
                                       min_samples_split=5, random_state=42))])


In [ ]:
# ========= MÉTRICAS ========= #
r2 = r2_score(y_test, pred)
MAE = mean_absolute_error(y_test, pred)
RMSE= np.sqrt(mean_squared_error(y_test, pred))

print("R² del modelo:", r2)
print("MAE del modelo:", MAE)
print("RMSE del modelo:", RMSE)

R² del modelo: 0.546094983926598
MAE del modelo: 13622.555138765374
RMSE del modelo: 59508.478676720144


###4.2.2 Usando GridSearchCV

In [ ]:
# ========= FEATURES ========= #
'''features = [
    "brand",
    "model_year",
    "milage",
    "fuel_type",
    "clean_title",
    "price_group",
    "milage_group",
    "engine_hp",
    "engine_l"
]
'''
features = [col for col in df_muestra.columns if col not in ['price', 'accident', 'brand', 'fuel_type','id', 'model', 'engine', 'ext_col', 'int_col', 'transmission']]
#Quitamos esas columnas porque no las creemos importantes para este modelo.
#Se dejan arriba las columnas en comentario para saber de un vistazo cuál tenemos.


X = df_muestra[features] #variables con las que se quiere predecir
y = df_muestra["price"] #variable que se quiere predecir

cat_cols = ["clean_title", "price_group", "milage_group"]
num_cols = ["model_year", "milage", "engine_hp", "engine_l"]

print(f"Total muestras (filas): {len(X)}")
print(f"Características (columnas): {X.shape[1]}")

Total muestras (filas): 94266
Características (columnas): 7


In [ ]:
# ========= SPLIT ========= #
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#80% para entrenamiento y 20% para test
print(f"Train: {len(X_train)} muestras ({len(X_train)/len(X)*100:.0f}%)")
print(f"Test: {len(X_test)} muestras ({len(X_test)/len(X)*100:.0f}%)")

Train: 75412 muestras (80%)
Test: 18854 muestras (20%)


In [ ]:
# ========= PREPROCESADO ========= #
preprocesado = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
])

In [ ]:
# ========= MODELO ========= #
#Modelo base sin hiperparámetros a optimizar
modelo = Pipeline([
    ("prep", preprocesado),
    ("rf", RandomForestRegressor(
        random_state=42
    ))
])

In [ ]:
#Rejilla con hiperparámetros a ajustar
param_dist = {
    "rf__n_estimators": [100, 150],
    "rf__max_depth": [10, 20],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", "log2"]
}

In [ ]:
#Creación del GridSearchCV con validación cruzada

scoring = {
    'R2': 'r2',
    'RMSE': 'neg_root_mean_squared_error',
    'MAE': 'neg_mean_absolute_error'
}

grid_search = GridSearchCV(
    estimator=modelo,
    param_grid=param_dist,
    cv=3,
    scoring='r2',
    n_jobs=-1
)


In [ ]:
# Entrenar el modelo
grid_search.fit(X_train, y_train)


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('prep',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['model_year',
                                                                          'milage',
                                                                          'engine_hp',
                                                                          'engine_l']),
                                                                        ('cat',
                                                                         OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                        unknown_value=-1),
                                                                         ['clean_title',
                                                                          'price_group',
                                                                          'milage_group'])])),
                                       ('rf',
                                        RandomForestRegressor(random_state=42))]),
             n_jobs=-1,
             param_grid={'rf__max_depth': [10, 20],
                         'rf__max_features': ['sqrt', 'log2'],
                         'rf__min_samples_leaf': [1, 2],
                         'rf__min_samples_split': [2, 5],
                         'rf__n_estimators': [100, 150]},
             scoring='r2')

In [ ]:
# ========= PREDICCIONES ========= #
pred = grid_search.predict(X_test)

In [ ]:
# ========= MÉTRICAS ========= #
r2 = r2_score(y_test, pred)
MAE = mean_absolute_error(y_test, pred)
RMSE= np.sqrt(mean_squared_error(y_test, pred))

print(grid_search.best_estimator_)
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor score R²:", grid_search.best_score_)
print("MAE del modelo:", MAE)
print("RMSE del modelo:", RMSE)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['model_year', 'milage',
                                                   'engine_hp', 'engine_l']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['clean_title', 'price_group',
                                                   'milage_group'])])),
                ('rf',
                 RandomForestRegressor(max_depth=10, max_features='sqrt',
                                       min_samples_split=5, random_state=42))])
Mejores parámetros: {'rf__max_depth': 10, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 5, 'rf__n_estimators': 100}
Mejor score R²: 0.6198599

#5. Exportamos el modelo

Exportamos el modelo con los mejores parámetros para poder usarlos de una manera más interactiva ya sea con powerBI o streamlit.

In [ ]:
# El mejor modelo (pipeline) ya está entrenado y disponible en random_search.best_estimator_
best_model_pipeline = random_search.best_estimator_

# Guardamos el modelo para streamlit
joblib.dump(best_model_pipeline, "best_model_sample.pkl")

print("Modelo guardado como 'best_model_sample.pkl'")

Modelo guardado como 'best_model_sample.pkl'


In [ ]:
# Guardamos df_muestra para poder usarlo en PowerBI si es necesario
df_muestra.to_csv("modelop7.csv", index=False)
print("DataFrame 'df_muestra' guardado como 'modelop7.csv'")

DataFrame 'df_muestra' guardado como 'modelop7.csv'
